<a href="https://colab.research.google.com/github/samuelaojih/Google-Colab/blob/main/CHIRPS_Rainfall_Statistics_Ilaje_LGA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==============================================================
# CHIRPS ANNUAL RAINFALL STATISTICS - ILAJE LGA
# Zonal Max / Min / Std Dev / Mean of Annual Rainfall Totals
# ==============================================================
#
# This script performs the following:
# 1. Connects to Google Earth Engine
# 2. Loads the study area asset (Ilaje LGA) as the region of interest
# 3. Loads CHIRPS Daily rainfall data (UCSB-CHG/CHIRPS/DAILY)
# 4. For each target year, sums daily rainfall into an annual total
#    rainfall image (mm/year) per pixel
# 5. Computes zonal statistics (max, min, mean, std dev) of that
#    annual total across the study area
# 6. Assembles the results into a pandas DataFrame
# 7. Exports the results table to CSV (locally and optionally to Drive)
#
# Target years: 1986, 1996, 2006, 2016, 2026
# Note: CHIRPS Daily data starts in 1981, so 1986-2016 are complete
# calendar years. 2026 is the current year at time of writing, so its
# annual total will only reflect the days available in the CHIRPS
# archive so far (CHIRPS also has a ~1-2 month latency for final data).
# ==============================================================

# ==============================================================
# STEP 1: INSTALL REQUIRED PACKAGES (Run in Colab if needed)
# ==============================================================

!pip install -U geemap


In [ ]:
# ==============================================================
# STEP 2: IMPORT LIBRARIES
# ==============================================================

import ee
import geemap
import pandas as pd


In [ ]:
# ==============================================================
# STEP 3: AUTHENTICATE AND INITIALIZE EARTH ENGINE
# ==============================================================

# Authenticate once (will prompt login)
ee.Authenticate()

# Initialize using your Earth Engine project ID
ee.Initialize(project='ee-samuelcoolsdk')


In [ ]:
# ==============================================================
# STEP 4: LOAD STUDY AREA
# ==============================================================

STUDY_AREA_ASSET = "projects/ee-samuelcoolsdk/assets/Ilaje_LGA"

study_area = ee.FeatureCollection(STUDY_AREA_ASSET)
roi = study_area.geometry()

print("Study area loaded:", STUDY_AREA_ASSET)
print("Area (sq km):", roi.area().divide(1e6).getInfo())


In [ ]:
# ==============================================================
# STEP 5: DEFINE CHIRPS COLLECTION AND TARGET YEARS
# ==============================================================

CHIRPS_COLLECTION = "UCSB-CHG/CHIRPS/DAILY"
CHIRPS_SCALE = 5566  # native CHIRPS resolution (~0.05 deg) in meters

chirps = ee.ImageCollection(CHIRPS_COLLECTION)

target_years = [1986, 1996, 2006, 2016, 2026]


In [ ]:
# ==============================================================
# STEP 6: FUNCTION TO COMPUTE ANNUAL RAINFALL ZONAL STATISTICS
# ==============================================================

def get_annual_rainfall_stats(year, geometry, scale=CHIRPS_SCALE):
    """
    Sums CHIRPS daily precipitation into an annual total image (mm/year)
    for the given year, then computes max, min, mean and std dev of that
    annual total across `geometry`.
    """
    start = ee.Date.fromYMD(year, 1, 1)
    end = start.advance(1, "year")

    annual_collection = chirps.filterDate(start, end).filterBounds(geometry)
    n_images = annual_collection.size().getInfo()

    # Annual total rainfall (mm/year) per pixel
    annual_total = annual_collection.select("precipitation").sum().clip(geometry)

    reducers = (
        ee.Reducer.minMax()
        .combine(reducer2=ee.Reducer.mean(), sharedInputs=True)
        .combine(reducer2=ee.Reducer.stdDev(), sharedInputs=True)
    )

    stats = annual_total.reduceRegion(
        reducer=reducers,
        geometry=geometry,
        scale=scale,
        maxPixels=1e13,
        bestEffort=True,
    ).getInfo()

    return {
        "year": year,
        "images_used": n_images,
        "max_mm": stats.get("precipitation_max"),
        "min_mm": stats.get("precipitation_min"),
        "mean_mm": stats.get("precipitation_mean"),
        "std_mm": stats.get("precipitation_stdDev"),
    }


In [ ]:
# ==============================================================
# STEP 7: RUN FOR EACH TARGET YEAR
# ==============================================================

results = []
for yr in target_years:
    print(f"Processing year {yr} ...")
    results.append(get_annual_rainfall_stats(yr, roi))

rainfall_df = pd.DataFrame(results)
rainfall_df = rainfall_df[["year", "images_used", "max_mm", "min_mm", "mean_mm", "std_mm"]]
rainfall_df


**Note on 2026:** CHIRPS Daily is only populated up to the latest processed date, and the row for 2026 will therefore be a partial-year total (not a full calendar year), with `images_used` showing how many days were actually available. Re-run the notebook later in the year, or after CHIRPS has released its 2026 final data, to get a complete annual figure.

In [ ]:
# ==============================================================
# STEP 8: EXPORT RESULTS TO CSV
# ==============================================================

output_csv = "Ilaje_LGA_CHIRPS_Annual_Rainfall_Stats.csv"
rainfall_df.to_csv(output_csv, index=False)
print(f"Saved: {output_csv}")

# Optional: save a copy to Google Drive (uncomment to use in Colab)
# from google.colab import drive
# drive.mount('/content/drive')
# rainfall_df.to_csv(f"/content/drive/MyDrive/{output_csv}", index=False)


In [ ]:
# ==============================================================
# STEP 9 (OPTIONAL): QUICK BAR CHART OF ANNUAL MEAN RAINFALL
# ==============================================================

import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(rainfall_df["year"].astype(str), rainfall_df["mean_mm"], color="#3182bd")
ax.set_xlabel("Year")
ax.set_ylabel("Mean Annual Rainfall (mm)")
ax.set_title("CHIRPS Mean Annual Rainfall - Ilaje LGA")
plt.tight_layout()
plt.show()
